# Mi perfil de frecuencias GTFS

Consulté frequencies y sus referencias el 12 de noviembre de 2026.
Conservo nueve consultas correctas de lectura. Recuperé 77 filas estructuradas completas.
El [perfil](../../../docs/frequencies_profile_2025_2026.md) explica los hallazgos.
El [contrato implementado](../../../docs/contracts/frequencies.md) define tipos y controles. La primera publicación Silver sigue pendiente.

Una fila define una ventana de frecuencia para un viaje plantilla.
No cuenta vehículos observados ni garantiza salidas exactas.

Las consultas no publican tablas ni forman parte del Job.
Todavía no ejecuté este cuaderno completo como notebook. Ejecuté sus consultas mediante Genie.
Los nombres y hashes son valores actuales del proyecto.
Las lecturas exploratorias no fijan versiones Delta ni una transacción conjunta.
Las consultas iniciales filtran hashes y muestran versiones. Comprobé que los resultados corresponden a las parejas esperadas.
Los bloques posteriores filtran explícitamente versión y hash.


## Bloque 1 — Esquema de entrada y dependencia

Reviso 27 campos de metadata, frequencies Bronze y trips Silver. Los cuatro campos de negocio de Bronze son string.


In [ ]:
%sql
-- BLOQUE 1 — Perfil de solo lectura.
SELECT 
  table_catalog,
  table_schema,
  table_name,
  column_name,
  ordinal_position,
  data_type
FROM workspace.information_schema.columns
WHERE table_schema = 'gtfs_bogota'
  AND table_name IN ('bronze_gtfs_frequencies', 'silver_gtfs_trips', 'audit_gtfs_source_schema_observations')
ORDER BY table_name, ordinal_position;


## Bloque 2 — Encabezados publicados

Reviso dos observaciones de frequencies.txt. Ambos archivos publican cuatro campos. Ninguno publica exact_times.


In [ ]:
%sql
-- BLOQUE 2 — Perfil de solo lectura.
SELECT 
  `source_feed_version`,
  `source_sha256`,
  `source_file_name`,
  `source_columns_json`,
  `observed_at_utc`
FROM `workspace`.`gtfs_bogota`.`audit_gtfs_source_schema_observations`
WHERE `source_file_name` = 'frequencies.txt'
  AND `source_sha256` IN (
    '07ba2ba9bb3b5bc1e38357355956757ce84fc68e6d05b74577bf017e189ab952',
    '08d14e0b3a5b9caa5451b722659871fd554e3273b94bdd770fdf81a26bf02097'
  )
ORDER BY `source_feed_version`, `source_sha256`;


## Bloque 3 — Trips publicada

Compruebo 107.162 / 181.060 filas y claves únicas. Ambos snapshots usan trips_v1. Esta lectura complementa el cierre funcional anterior.


In [ ]:
%sql
-- BLOQUE 3 — Perfil de solo lectura.
SELECT 
  `source_feed_version`,
  `source_sha256`,
  COUNT(*) as total_rows,
  COUNT(DISTINCT `trip_id`) as distinct_trip_ids,
  COUNT(DISTINCT `silver_rule_version`) as distinct_rule_versions,
  COLLECT_SET(`silver_rule_version`) as rule_version_values
FROM `workspace`.`gtfs_bogota`.`silver_gtfs_trips`
WHERE `source_sha256` IN (
    '07ba2ba9bb3b5bc1e38357355956757ce84fc68e6d05b74577bf017e189ab952',
    '08d14e0b3a5b9caa5451b722659871fd554e3273b94bdd770fdf81a26bf02097'
  )
GROUP BY `source_feed_version`, `source_sha256`
ORDER BY `source_feed_version`;


## Bloque 4 — Ventanas y procedencia

Recupero las 16 filas fuente, con un límite de 100. Son 2 / 14 ventanas. No guardo resultados dentro del notebook.


In [ ]:
%sql
-- BLOQUE 4 — Perfil de solo lectura.
SELECT 
  `trip_id`,
  `start_time`,
  `end_time`,
  `headway_secs`,
  `source_feed_version`,
  `source_sha256`,
  `source_file_path`,
  `bronze_loaded_at_utc`
FROM `workspace`.`gtfs_bogota`.`bronze_gtfs_frequencies`
WHERE `source_sha256` IN (
    '07ba2ba9bb3b5bc1e38357355956757ce84fc68e6d05b74577bf017e189ab952',
    '08d14e0b3a5b9caa5451b722659871fd554e3273b94bdd770fdf81a26bf02097'
  )
ORDER BY `source_feed_version`, `trip_id`, `start_time`
LIMIT 100;


## Bloque 5 — Completitud por campo

Reviso ocho métricas por versión y campo. No encuentro nulos, vacíos, espacios extremos ni caracteres de control.


In [ ]:
%sql
-- BLOQUE 5 — Perfil de solo lectura.
with source as (
 select trip_id,start_time,end_time,headway_secs,source_feed_version,source_sha256
 from workspace.gtfs_bogota.bronze_gtfs_frequencies where ((source_feed_version='2025-10-28' and source_sha256='07ba2ba9bb3b5bc1e38357355956757ce84fc68e6d05b74577bf017e189ab952') or (source_feed_version='2026-09-29' and source_sha256='08d14e0b3a5b9caa5451b722659871fd554e3273b94bdd770fdf81a26bf02097'))
), fields as (
 select source_feed_version,source_sha256,field_name,value
 from source lateral view stack(4,'trip_id',trip_id,'start_time',start_time,'end_time',end_time,'headway_secs',headway_secs) u as field_name,value
)
select source_feed_version,source_sha256,field_name,count(*) as snapshot_rows,
 count_if(value is null) as null_count,count_if(value='') as empty_count,
 count_if(value<>'' and value rlike r'^[\s\p{Z}]*$') as whitespace_only_count,
 count_if(value rlike r'^[\s\p{Z}]|[\s\p{Z}]$') as edge_whitespace_count,
 count_if(value rlike r'[\p{Cc}\p{Cf}]') as control_count,
 count(distinct case when value is not null and not(value rlike r'^[\s\p{Z}]*$') then value end) as distinct_non_blank_count
from fields group by source_feed_version,source_sha256,field_name
order by source_feed_version,field_name;


## Bloque 6 — Conversión, rangos y procedencia

Compruebo dos resúmenes. Las conversiones, ventanas, intervalos y procedencia revisados no presentan errores. No aparecen horas superiores a 24.


In [ ]:
%sql
-- BLOQUE 6 — Perfil de solo lectura.
with source as (
  select trip_id,start_time,end_time,headway_secs,source_feed_version,source_sha256,source_file_path,bronze_loaded_at_utc
  from workspace.gtfs_bogota.bronze_gtfs_frequencies
  where ((source_feed_version='2025-10-28' and source_sha256='07ba2ba9bb3b5bc1e38357355956757ce84fc68e6d05b74577bf017e189ab952') or (source_feed_version='2026-09-29' and source_sha256='08d14e0b3a5b9caa5451b722659871fd554e3273b94bdd770fdf81a26bf02097'))
), parsed as (
  select source.*,case when start_time rlike '^[0-9]+:[0-5][0-9]:[0-5][0-9]$' then try_cast(try_cast(element_at(split(start_time,':'),1) as decimal(20,0))*3600 + try_cast(element_at(split(start_time,':'),2) as int)*60 + try_cast(element_at(split(start_time,':'),3) as int) as bigint) end as start_seconds,
    case when end_time rlike '^[0-9]+:[0-5][0-9]:[0-5][0-9]$' then try_cast(try_cast(element_at(split(end_time,':'),1) as decimal(20,0))*3600 + try_cast(element_at(split(end_time,':'),2) as int)*60 + try_cast(element_at(split(end_time,':'),3) as int) as bigint) end as end_seconds,
    case when headway_secs rlike '^[0-9]+$' then try_cast(headway_secs as int) end as headway_int
  from source
)
select source_feed_version,source_sha256,count(*) as rows,
 count(distinct trip_id) as trips,count(distinct struct(trip_id,start_time)) as raw_keys,
 count(distinct struct(trip_id,start_seconds)) as parsed_keys,
 count_if(start_seconds is null) as bad_start,count_if(end_seconds is null) as bad_end,
 count_if(headway_int is null or headway_int<=0) as bad_headway,
 count_if(end_seconds<=start_seconds) as bad_window,
 count_if(start_seconds>=86400 or end_seconds>=86400) as beyond_midnight,
 min(start_seconds) as min_start_seconds,max(end_seconds) as max_end_seconds,
 min(headway_int) as min_headway,max(headway_int) as max_headway,
 min(end_seconds-start_seconds) as min_duration_seconds,max(end_seconds-start_seconds) as max_duration_seconds,
 count_if(bronze_loaded_at_utc is null) as missing_loaded_at,
 count_if(source_file_path is null or not endswith(source_file_path,concat('/feed_version=',source_feed_version,'/sha256=',source_sha256,'/frequencies.txt'))) as bad_path
from parsed group by source_feed_version,source_sha256 order by source_feed_version;


## Bloque 7 — Copias, claves y superposiciones

Compruebo dos resúmenes. No encuentro copias, conflictos ni superposiciones dentro de un mismo trip_id. Cada viaje actual tiene una ventana.


In [ ]:
%sql
-- BLOQUE 7 — Perfil de solo lectura.
with source as (
  select trip_id,start_time,end_time,headway_secs,source_feed_version,source_sha256,source_file_path,bronze_loaded_at_utc
  from workspace.gtfs_bogota.bronze_gtfs_frequencies
  where ((source_feed_version='2025-10-28' and source_sha256='07ba2ba9bb3b5bc1e38357355956757ce84fc68e6d05b74577bf017e189ab952') or (source_feed_version='2026-09-29' and source_sha256='08d14e0b3a5b9caa5451b722659871fd554e3273b94bdd770fdf81a26bf02097'))
), parsed as (
  select source.*,case when start_time rlike '^[0-9]+:[0-5][0-9]:[0-5][0-9]$' then try_cast(try_cast(element_at(split(start_time,':'),1) as decimal(20,0))*3600 + try_cast(element_at(split(start_time,':'),2) as int)*60 + try_cast(element_at(split(start_time,':'),3) as int) as bigint) end as start_seconds,
    case when end_time rlike '^[0-9]+:[0-5][0-9]:[0-5][0-9]$' then try_cast(try_cast(element_at(split(end_time,':'),1) as decimal(20,0))*3600 + try_cast(element_at(split(end_time,':'),2) as int)*60 + try_cast(element_at(split(end_time,':'),3) as int) as bigint) end as end_seconds,
    case when headway_secs rlike '^[0-9]+$' then try_cast(headway_secs as int) end as headway_int
  from source
), business as (
 select distinct source_feed_version,source_sha256,trip_id,start_time,end_time,headway_secs,start_seconds,end_seconds from parsed
), raw_keys as (
 select source_feed_version,source_sha256,trip_id,start_time,count(*) as variants from business
 group by source_feed_version,source_sha256,trip_id,start_time
), parsed_keys as (
 select source_feed_version,source_sha256,trip_id,start_seconds,count(*) as variants from business
 group by source_feed_version,source_sha256,trip_id,start_seconds
), ordered as (
 select *,max(end_seconds) over(partition by source_feed_version,source_sha256,trip_id order by start_seconds,end_seconds,start_time,headway_secs rows between unbounded preceding and 1 preceding) as prior_max_end
 from business where start_seconds is not null and end_seconds>start_seconds
), copies as (
 select source_feed_version,source_sha256,trip_id,start_time,end_time,headway_secs,count(*) as copies
 from parsed group by source_feed_version,source_sha256,trip_id,start_time,end_time,headway_secs
), versions as (select distinct source_feed_version,source_sha256 from source)
select v.source_feed_version,v.source_sha256,
 (select coalesce(sum(c.copies-1),0) from copies c where c.source_feed_version=v.source_feed_version and c.source_sha256=v.source_sha256) as identical_extra_copies,
 (select count(*) from raw_keys k where k.source_feed_version=v.source_feed_version and k.source_sha256=v.source_sha256 and k.variants>1) as raw_key_conflicts,
 (select count(*) from parsed_keys k where k.source_feed_version=v.source_feed_version and k.source_sha256=v.source_sha256 and k.variants>1) as parsed_key_conflicts,
 (select count(*) from ordered o where o.source_feed_version=v.source_feed_version and o.source_sha256=v.source_sha256 and o.start_seconds<o.prior_max_end) as overlapping_rows,
 (select count(*) from ordered o where o.source_feed_version=v.source_feed_version and o.source_sha256=v.source_sha256 and o.start_seconds=o.prior_max_end) as touching_rows
from versions v order by v.source_feed_version;


## Bloque 8 — Relación con trips Silver

Compruebo dos resúmenes. El join conserva 2 / 14 ventanas. Todas las referencias resuelven dentro del mismo snapshot.


In [ ]:
%sql
-- BLOQUE 8 — Perfil de solo lectura.
with source as (
 select trip_id,start_time,source_feed_version,source_sha256 from workspace.gtfs_bogota.bronze_gtfs_frequencies where ((source_feed_version='2025-10-28' and source_sha256='07ba2ba9bb3b5bc1e38357355956757ce84fc68e6d05b74577bf017e189ab952') or (source_feed_version='2026-09-29' and source_sha256='08d14e0b3a5b9caa5451b722659871fd554e3273b94bdd770fdf81a26bf02097'))
), trips as (
 select trip_id,route_id,service_id,shape_id,silver_rule_version,source_feed_version,source_sha256
 from workspace.gtfs_bogota.silver_gtfs_trips where ((source_feed_version='2025-10-28' and source_sha256='07ba2ba9bb3b5bc1e38357355956757ce84fc68e6d05b74577bf017e189ab952') or (source_feed_version='2026-09-29' and source_sha256='08d14e0b3a5b9caa5451b722659871fd554e3273b94bdd770fdf81a26bf02097'))
)
select f.source_feed_version,f.source_sha256,count(*) as joined_rows,
 count_if(t.trip_id is null) as missing_trip_rows,
 count_if(t.trip_id is not null and not(t.silver_rule_version<=>'trips_v1')) as wrong_rule_rows,
 count(distinct t.route_id) as routes,count(distinct t.service_id) as services,count(distinct t.shape_id) as shapes
from source f left join trips t on f.source_feed_version=t.source_feed_version and f.source_sha256=t.source_sha256 and f.trip_id=t.trip_id
 group by f.source_feed_version,f.source_sha256 order by f.source_feed_version;


## Bloque 9 — Contexto de ruta y calendario

Recupero 16 filas de contexto. Todas corresponden a C_1, con route_type=6. El calendario explica patrones semanales; las excepciones requieren calendar_dates.


In [ ]:
%sql
-- BLOQUE 9 — Perfil de solo lectura.
with source as (
 select trip_id,start_time,end_time,headway_secs,source_feed_version,source_sha256
 from workspace.gtfs_bogota.bronze_gtfs_frequencies where ((source_feed_version='2025-10-28' and source_sha256='07ba2ba9bb3b5bc1e38357355956757ce84fc68e6d05b74577bf017e189ab952') or (source_feed_version='2026-09-29' and source_sha256='08d14e0b3a5b9caa5451b722659871fd554e3273b94bdd770fdf81a26bf02097'))
), trips as (
 select trip_id,route_id,service_id,source_feed_version,source_sha256 from workspace.gtfs_bogota.silver_gtfs_trips where ((source_feed_version='2025-10-28' and source_sha256='07ba2ba9bb3b5bc1e38357355956757ce84fc68e6d05b74577bf017e189ab952') or (source_feed_version='2026-09-29' and source_sha256='08d14e0b3a5b9caa5451b722659871fd554e3273b94bdd770fdf81a26bf02097'))
), routes as (
 select route_id,route_type,route_short_name,route_long_name,source_feed_version,source_sha256 from workspace.gtfs_bogota.silver_gtfs_routes where ((source_feed_version='2025-10-28' and source_sha256='07ba2ba9bb3b5bc1e38357355956757ce84fc68e6d05b74577bf017e189ab952') or (source_feed_version='2026-09-29' and source_sha256='08d14e0b3a5b9caa5451b722659871fd554e3273b94bdd770fdf81a26bf02097'))
), calendar as (
 select service_id,monday,tuesday,wednesday,thursday,friday,saturday,sunday,start_date,end_date,source_feed_version,source_sha256 from workspace.gtfs_bogota.silver_gtfs_calendar where ((source_feed_version='2025-10-28' and source_sha256='07ba2ba9bb3b5bc1e38357355956757ce84fc68e6d05b74577bf017e189ab952') or (source_feed_version='2026-09-29' and source_sha256='08d14e0b3a5b9caa5451b722659871fd554e3273b94bdd770fdf81a26bf02097'))
)
select f.source_feed_version,f.source_sha256,f.trip_id,f.start_time,f.end_time,f.headway_secs,
 t.route_id,t.service_id,r.route_type,r.route_short_name,r.route_long_name,
 c.monday,c.tuesday,c.wednesday,c.thursday,c.friday,c.saturday,c.sunday,c.start_date,c.end_date
from source f left join trips t on f.source_feed_version=t.source_feed_version and f.source_sha256=t.source_sha256 and f.trip_id=t.trip_id
left join routes r on t.source_feed_version=r.source_feed_version and t.source_sha256=r.source_sha256 and t.route_id=r.route_id
left join calendar c on t.source_feed_version=c.source_feed_version and t.source_sha256=c.source_sha256 and t.service_id=c.service_id
order by f.source_feed_version,f.trip_id,f.start_time limit 100;
